# 10年定着予測 - カテゴリ専用 / 数値専用モデルのアンサンブル（46_）

**背景**: `43_` でモデル多様性（LightGBM・XGBoost）を試したが、
**多様性はあったのに（相関0.92〜0.94）代替モデルが弱すぎて**（+0.026〜+0.032）
提出候補ゼロで終わった。

そこで**モデルファミリは CatBoost のまま、入力を分割することで多様性を作る**。

| モデル | 入力 |
|---|---|
| `M0_lean113` | **現最良の113列**（`40_` R6_lean, Public 0.521729） |
| `M1_cat` | **カテゴリ列のみ** |
| `M2_num` | **数値列のみ** |

`M1` と `M2` は入力が**互いに素**なので、予測が似る理由が構造的に無い。
`43_` が失敗した「多様性はあるが弱い」ではなく、
**「意図的に情報を落として多様性を作る」**アプローチである。

## 特徴量プールは却下ブロックを全部含む

`M1`・`M2` の候補列は、**これまで却下されたブロックをすべて含む**（ユーザ指示）。
`39_` が移植済みの9ブロックをそのまま使う。

| ブロック | 内容 | 過去の判定 |
|---|---|---|
| E | メモ構造化（キャリア志向・転居許容・在宅希望・希望勤務地） | Publicで転移 |
| F | 経験等級整合性（学習期間IDのみで線形回帰） | 却下 |
| G | 自己学習 | 却下（2度） |
| H | エンゲージメント深掘り | 却下 |
| I | 人物所見キーワード | 却下 |
| J | 希望勤務地マッチ | Publicで転移 |
| K | 昇給タイミング | 却下 |
| M | 専攻×職種ミスマッチ | 却下 |
| N | モメンタム | 却下 |
| O | 活動密度 | 却下 |
| L2 | 転居×勤務地ミスマッチ | **現行採用** |

**却下されたのは「441列に足したとき」であって、
「カテゴリだけ／数値だけの小さなモデルの中で」使えないとは限らない。**

## 重要度で絞る（ユーザ承認済み）

プールが大きいままだと `42_` の教訓（列が多すぎると悪化）に逆行するので、
**CatBoost の特徴量重要度で上位を残す**。

事前登録した規則:

1. 80%学習データでプール全列を使って CatBoost を1本学習（`A_PARAMS`・560反復・seed=42）
2. `PredictionValuesChange` 重要度を降順に並べ、**累積が全体の95%に達するまで**を採用
3. ただし**最小10列・最大100列**にクリップする

**重要**: この重要度は**検証セットを一切使わない**（学習データの構造だけから計算される）。
`40_` のLOO診断（検証スコアで選び、`42_` で符号反転）とは性質が違う。

## 実行構成

| config | 内容 |
|---|---|
| `M0_lean113` | 参照。`R6_lean` の再現。**提出しない** |
| `M1_cat` | カテゴリ専用（重要度で選抜） |
| `M2_num` | 数値専用（重要度で選抜） |
| `E_M0_M1` / `E_M0_M2` / `E_M1_M2` / `E_ALL3` | **等重み平均**（重みは学習しない） |

## ゲート（事前登録・`43_` と同一）

- **参加ゲート**: 単体valが `M0` から **+0.01 以内**のモデルだけアンサンブルに入れる
- **提出ゲート**: `M0` との予測平均絶対差が **ノイズ床（`41_` 実測: 95%上限 0.02122）** を超えるもののみ
- `M1`・`M2` の単体は**提出しない**（材料の質を測るためだけ）

ゲートで全滅しても、`43_` と同様に**「ゲートを無視して平均していたらどうなったか」を必ず出力する**。
そうすれば失敗した場合も情報が残る。

## 実行環境

Colab Pro CPUハイメモリ。想定 **40〜60分**。

> ⚠️ **ローカルMacで先行実行しないこと。**


In [27]:
!pip install -q catboost optuna

In [28]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [29]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [31]:
SCRIPT_NAME = "46_cat_num_split_ensemble"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# チェックポイント（日付非依存の固定パス。セッションをまたいで再開できるようにする）
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

RESET_CHECKPOINT = True  # Trueにすると既存チェックポイントを削除して最初から再計算する
if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("チェックポイントを削除しました（全構成を再計算します）")

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")

[2026-08-12 13:10:19] [INFO] === [46_cat_num_split_ensemble] 実験開始 ===


INFO:46_cat_num_split_ensemble:=== [46_cat_num_split_ensemble] 実験開始 ===


[2026-08-12 13:10:19] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260812


INFO:46_cat_num_split_ensemble:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260812


[2026-08-12 13:10:19] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/46_cat_num_split_ensemble_checkpoint.csv


INFO:46_cat_num_split_ensemble:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/46_cat_num_split_ensemble_checkpoint.csv


[2026-08-12 13:10:19] [INFO] チェックポイントは未作成（新規実行）


INFO:46_cat_num_split_ensemble:チェックポイントは未作成（新規実行）


In [32]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-12 13:10:20] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:46_cat_num_split_ensemble:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-12 13:10:20] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:46_cat_num_split_ensemble:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-12 13:10:20] [INFO] 定着率: 0.5647


INFO:46_cat_num_split_ensemble:定着率: 0.5647


[2026-08-12 13:10:20] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:46_cat_num_split_ensemble:Train IDs: 2761, Test IDs: 2502


## 0. 早期退職者の特定（改善3の前提）

`月末在籍状態 == "退職"` の行を持つ社員が、0-23ヶ月の観測期間中に退職した社員。
Train に129名（全員ラベル0）、**Test には0名**。

In [33]:
EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())

logger.info(f"Train 早期退職者: {len(EARLY_LEAVER_IDS)}名 / {len(train_ids)}名 ({len(EARLY_LEAVER_IDS)/len(train_ids):.1%})")
logger.info(f"Test  早期退職者: {len(_test_early)}名 / {len(test_ids)}名")
_y_idx = train_persona.set_index(ID_COL)[TARGET_COL]
logger.info(f"早期退職者のラベル平均: {_y_idx.loc[list(EARLY_LEAVER_IDS)].mean():.4f}（0.0のはず）")
logger.info(f"定着率: 全体 {y_train.mean():.4f} / 早期退職者を除く {_y_idx[~_y_idx.index.isin(EARLY_LEAVER_IDS)].mean():.4f}")
assert len(_test_early) == 0, "Testに早期退職者が存在する。EDA v6の前提が崩れているので調査すること"

[2026-08-12 13:10:20] [INFO] Train 早期退職者: 129名 / 2761名 (4.7%)


INFO:46_cat_num_split_ensemble:Train 早期退職者: 129名 / 2761名 (4.7%)


[2026-08-12 13:10:20] [INFO] Test  早期退職者: 0名 / 2502名


INFO:46_cat_num_split_ensemble:Test  早期退職者: 0名 / 2502名


[2026-08-12 13:10:20] [INFO] 早期退職者のラベル平均: 0.0000（0.0のはず）


INFO:46_cat_num_split_ensemble:早期退職者のラベル平均: 0.0000（0.0のはず）


[2026-08-12 13:10:20] [INFO] 定着率: 全体 0.5647 / 早期退職者を除く 0.5923


INFO:46_cat_num_split_ensemble:定着率: 全体 0.5647 / 早期退職者を除く 0.5923


## 1. 基本特徴量関数の定義（split非依存、`18_`〜`26_`と同一ロジック）

In [34]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")

✅ split非依存の基本特徴量関数定義完了


In [35]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")

[2026-08-12 13:10:21] [INFO] ------------------------------------------------------------


INFO:46_cat_num_split_ensemble:------------------------------------------------------------


[2026-08-12 13:10:21] [INFO] split非依存の基本特徴量を生成中...


INFO:46_cat_num_split_ensemble:split非依存の基本特徴量を生成中...


[2026-08-12 13:10:21] [INFO] ------------------------------------------------------------


INFO:46_cat_num_split_ensemble:------------------------------------------------------------


[2026-08-12 13:16:05] [INFO] split非依存の基本特徴量生成完了


INFO:46_cat_num_split_ensemble:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`18_`と同一・継続採用）

In [36]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")

[2026-08-12 13:16:06] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:46_cat_num_split_ensemble:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-12 13:16:07] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:46_cat_num_split_ensemble:入社時メモ: SVD累積寄与率=0.760


[2026-08-12 13:16:11] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:46_cat_num_split_ensemble:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-12 13:16:13] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:46_cat_num_split_ensemble:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-12 13:16:13] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:46_cat_num_split_ensemble:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`18_`の勝者を継続採用）

`18_`のステップAで、D_expanded（16指標）がD_original（6指標）・Dなしより2 split平均で最良と判明したため、
以降は常にD_expandedを使う（今回はDブロックの再比較は行わない）。

In [37]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")

[2026-08-12 13:16:13] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:46_cat_num_split_ensemble:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-12 13:18:15] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:46_cat_num_split_ensemble:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（split非依存、`18_`と同一）

In [38]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")

[2026-08-12 13:18:15] [INFO] Persona単位の基本特徴量を生成中...


INFO:46_cat_num_split_ensemble:Persona単位の基本特徴量を生成中...


[2026-08-12 13:18:15] [INFO] Persona単位の基本特徴量処理完了


INFO:46_cat_num_split_ensemble:Persona単位の基本特徴量処理完了


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、v1=`27_`のPublic確認済み版 / v2=抽出拡張版）

`転居許容`フラグの抽出ロジックは`27_`と同一。`希望勤務地`の抽出のみ2種類を用意する：

- **v1**: `27_`・`25_`・`data_exploration_v3/v4/v5`と同一の正規表現（Public 0.529672で確認済み）
- **v2**: v1に加え、「◯◯を希望。」「◯◯勤務を希望。」「◯◯での勤務を希望。」パターンを追加で
  拾う拡張版。未抽出だった152件（train）を目視確認して発見した言い回し。カバー率が
  88.6%→94.1%（train）/ 95.0%（test）に向上し、ダブル悪条件の該当件数も342→354件に増加、
  効果量はp=1.4×10⁻³⁹→1.7×10⁻⁴³・オッズ比0.189→0.178とむしろ強まった。

In [39]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    return m.group(1).strip() if m else None


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    # reloc_ok_rawはobject dtype(True/False/None混在)のため、~演算子は使わず
    # 明示的な等価比較でTrue/False/欠損を扱う（欠損に対する~はTypeErrorになる）
    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    # 4値カテゴリ（決定木が交互作用を直接学習しやすいよう明示的にエンコード）
    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    # ダブル悪条件フラグ（EDA v5で確認した最も強いシグナル: 転居許容せず AND 勤務地不一致）
    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")
print("L_v1 ダブル悪条件:")
print(train_reloc_v1["転居x勤務地_ダブル悪条件_v1"].value_counts())
print("\nL_v2 ダブル悪条件:")
print(train_reloc_v2["転居x勤務地_ダブル悪条件_v2"].value_counts())

[2026-08-12 13:18:16] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


INFO:46_cat_num_split_ensemble:転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


[2026-08-12 13:18:16] [INFO] L_v1: Train (2761, 3), Test (2502, 3)


INFO:46_cat_num_split_ensemble:L_v1: Train (2761, 3), Test (2502, 3)


[2026-08-12 13:18:16] [INFO] L_v2: Train (2761, 3), Test (2502, 3)


INFO:46_cat_num_split_ensemble:L_v2: Train (2761, 3), Test (2502, 3)


L_v1 ダブル悪条件:
転居x勤務地_ダブル悪条件_v1
0    2419
1     342
Name: count, dtype: int64

L_v2 ダブル悪条件:
転居x勤務地_ダブル悪条件_v2
0    2407
1     354
Name: count, dtype: int64


## 6b. 部署Target Encoding（`15_`〜`37_`と同一）

In [40]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out

print("✅ 部署Target Encoding関数 定義完了")


✅ 部署Target Encoding関数 定義完了


In [41]:
RESULT_SCHEMA = ["config", "n_features", "val_score", "val_score_all", "val_score_single",
                   "val_single_mean", "val_single_sd", "best_iter", "n_iterations",
                   "params", "submission_path"]

def make_row(**kwargs):
    """全configで同じ列構成のdictを作る。

    28_ は全configが同じキーを持っていたが、37_ は A / BC / D で必要な情報が異なる。
    キー構成がバラバラのままだと、mode="a" でCSVに追記した際に列がずれて壊れるため、
    固定スキーマに揃えてから書き出す。
    """
    unknown = set(kwargs) - set(RESULT_SCHEMA)
    assert not unknown, f"RESULT_SCHEMAに無いキー: {unknown}"
    row = {k: np.nan for k in RESULT_SCHEMA}
    row.update(kwargs)
    return row


def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=RESULT_SCHEMA)

def save_checkpoint_row(result):
    df = pd.DataFrame([result])[RESULT_SCHEMA]
    write_header = not CHECKPOINT_PATH.exists()
    df.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)

def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label]
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元: val_score={row['val_score']}")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result

print("✅ チェックポイント関数定義完了（37_版: 固定スキーマで列ずれを防止）")

✅ チェックポイント関数定義完了（37_版: 固定スキーマで列ずれを防止）


In [42]:
SEEDS = [42, 2024, 7, 1234, 99]          # 改善2: シード平均に使う5シード
N_TRIALS = 25                            # 18_〜28_と同一
ITER_SCALE_CANDIDATES = {"x125": 1.25, "x100": 1.00}   # 全件学習時の反復数スケール（2761/2208≒1.25）


def _feature_cols(df):
    return [c for c in df.columns if c not in ["入社日", TARGET_COL]]


def _xy(df, feature_cols):
    return df[feature_cols].fillna(-999), df[TARGET_COL]


def tune_hyperparams(ag_train, ag_val, n_trials=N_TRIALS):
    """Optunaでハイパーパラメータを探索（探索空間は18_〜28_と完全に同一）"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)

    def objective(trial):
        params = {
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "iterations": 1000, "random_seed": SEED, "verbose": False,
            "cat_features": obj_cols, "early_stopping_rounds": 50, "task_type": "CPU",
        }
        model = cb.CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        return log_loss(y_va, model.predict_proba(X_va)[:, 1])

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    logger.info(f"  Optuna完了: best_value={study.best_value:.6f}, best_params={study.best_params}")
    return study.best_params


def fit_holdout(ag_train, ag_val, test_features, best_params, seeds):
    """80/20で学習。early stoppingで最良反復数を決め、シードごとの予測を返す"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)
    X_test = test_features[feature_cols].fillna(-999)

    val_preds, test_preds, best_iters = [], [], []
    for seed in seeds:
        model = cb.CatBoostClassifier(
            **best_params, iterations=3000, random_seed=seed, verbose=False,
            cat_features=obj_cols, early_stopping_rounds=100, task_type="CPU",
        )
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        vp = model.predict_proba(X_va)[:, 1]
        val_preds.append(vp)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        best_iters.append(model.get_best_iteration())
        logger.info(f"  seed={seed}: val_logloss={log_loss(y_va, vp):.6f}, best_iteration={best_iters[-1]}")

    return {
        "val_preds": np.array(val_preds), "test_preds": np.array(test_preds),
        "best_iters": best_iters, "y_val": y_va.values, "feature_cols": feature_cols,
    }


def fit_full_train(ag_full, test_features, best_params, n_iterations, seeds):
    """Train全件で学習（改善1）。検証セットが無いので反復数は固定、early stoppingなし"""
    feature_cols = _feature_cols(ag_full)
    obj_cols = [c for c in feature_cols if ag_full[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_full, feature_cols)
    X_test = test_features[feature_cols].fillna(-999)

    test_preds = []
    for seed in seeds:
        model = cb.CatBoostClassifier(
            **best_params, iterations=int(n_iterations), random_seed=seed, verbose=False,
            cat_features=obj_cols, task_type="CPU",
        )
        model.fit(X_tr, y_tr)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        logger.info(f"  seed={seed}: 全件学習完了（iterations={int(n_iterations)}）")
    return np.array(test_preds)


def save_submission(test_index, preds, config_label):
    path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    pd.DataFrame({ID_COL: test_index, TARGET_COL: preds}).to_csv(path, index=False, header=False)
    logger.info(f"  提出ファイル: {path.name}（予測平均={preds.mean():.4f}）")
    return str(path)


print("✅ モデル関数定義完了（tune_hyperparams / fit_holdout / fit_full_train）")

✅ モデル関数定義完了（tune_hyperparams / fit_holdout / fit_full_train）


## 20. 却下済み/未検証ブロックのビルダー関数

**すべて元のノートブックからそのまま移植している**（E/F/G: `20_`、H: `21_`、I: `24_`、J/K: `25_`、
M: `29_`、N/O: `34_`）。今回の目的は「同じ特徴量を、より低ノイズなプロトコルで測り直す」ことなので、
特徴量の中身には一切手を加えない。

`extract_workstyle_section` は `37_` のセル（ブロックL_v2用）で既に定義済みで、`20_`版と
完全に一致することを確認済みのため再定義しない。

In [43]:
# 各ブロックのビルダーは、それぞれを最初に導入したノートブックから**一字一句そのまま**移植している
# （E,F,G: 20_ / H: 21_ / I: 24_ / J,K: 25_ / M: 29_ / N,O: 34_）。
# 再検証の目的は「同じ特徴量を、より低ノイズなプロトコルで測り直す」ことなので、中身は変更しない。

from sklearn.linear_model import LinearRegression

_NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
_POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")
_NEG_REMOTE = re.compile(r"在宅勤務を(必須条件としていない|希望しない|希望していない|希望せず)")
_POS_REMOTE = re.compile(r"在宅勤務を希望")
ANALYTICAL_MAJOR = {"情報", "理工学"}
ANALYTICAL_JOB = {"IT・エンジニアリング", "データ・商品企画・コンサルティング"}

MOMENTUM_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

def extract_memo_section(text, section_name):
    if pd.isna(text):
        return None
    m = re.search(rf"{section_name}：(.+?)(?:\n|$)", text)
    return m.group(1).strip() if m else None


def extract_desired_location(s):
    if s is None:
        return "unknown"
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else "unknown"


def classify_career_orientation(s):
    if s is None:
        return "unknown"
    if "限定していない" in s or "限定しない" in s:
        return "未定"
    if "管理職" in s:
        return "管理職志向"
    if "専門職" in s:
        return "専門職志向"
    if "安定" in s:
        return "安定志向"
    return "other"


def classify_relocation(s):
    if s is None:
        return "unknown"
    if _NEG_RELOC.search(s):
        return "false"
    if _POS_RELOC.search(s):
        return "true"
    return "unknown"


def classify_remote_pref(s):
    if s is None:
        return "unknown"
    if _POS_REMOTE.search(s):
        return "true"
    if _NEG_REMOTE.search(s):
        return "false"
    return "unknown"


def create_memo_structured_features(persona_df):
    career_section = persona_df["入社時メモ"].apply(lambda t: extract_memo_section(t, "キャリア志向"))
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "memo_career_cat": career_section.apply(classify_career_orientation).values,
        "memo_転居許容": ws_section.apply(classify_relocation).values,
        "memo_在宅希望": ws_section.apply(classify_remote_pref).values,
        "memo_希望勤務地": ws_section.apply(extract_desired_location).values,
    })


def parse_all_study_themes(s):
    if pd.isna(s) or s == "受講なし":
        return [], 0.0
    parts = str(s).split("｜")
    themes, total = [], 0.0
    for part in parts:
        m = re.match(r"(.+?)：([\d.]+)時間", part)
        if m:
            themes.append(m.group(1))
            total += float(m.group(2))
    return themes, total


def create_self_study_features(monthly_df, employee_ids):
    """自己学習実施月数・合計時間・ユニークテーマ数を集計（EDA v3 分析7と同一ロジック）"""
    df = monthly_df[["社員ID", "自己学習（詳細）"]].copy()
    parsed = df["自己学習（詳細）"].apply(parse_all_study_themes)
    df["_themes"] = parsed.apply(lambda x: x[0])
    df["_hours"] = parsed.apply(lambda x: x[1])

    total_hours = df.groupby("社員ID")["_hours"].sum()
    active_months = df[df["_hours"] > 0].groupby("社員ID").size()
    unique_themes = df.groupby("社員ID")["_themes"].apply(lambda s: len(set(t for tl in s for t in tl)))

    out = pd.DataFrame({
        "自己学習合計時間": total_hours,
        "自己学習実施月数": active_months,
        "自己学習ユニークテーマ数": unique_themes,
    })
    out = out.reindex(employee_ids).fillna(0.0).reset_index().rename(columns={"index": "社員ID"})
    return out


def create_engagement_deepdive_features(monthly_df, employee_ids):
    other_eval_cols = ["360度評価_親和度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        info_vals = emp_data["情報共有件数"].values
        is_zero = info_vals == 0
        features["情報共有件数_ゼロ月数"] = int(is_zero.sum())
        max_run = cur_run = 0
        for v in is_zero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["情報共有件数_最長ゼロ連続月数"] = max_run

        trust_mean = emp_data["360度評価_信頼度"].mean()
        other_mean = emp_data[other_eval_cols].mean(axis=1).mean()
        features["360度評価_信頼度_相対偏差"] = (
            trust_mean - other_mean if pd.notna(trust_mean) and pd.notna(other_mean) else np.nan
        )

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_personal_impression_features(persona_df):
    obs_section = persona_df["入社時メモ"].apply(lambda t: extract_memo_section(t, "人物所見"))
    text = obs_section.fillna("")
    flex = text.apply(lambda t: any(k in t for k in ["柔軟", "切り替え", "適応"]))
    proactive = text.apply(lambda t: any(k in t for k in ["相談", "自ら", "主体的"]))
    plan = text.apply(lambda t: any(k in t for k in ["優先順位", "完了条件", "着実に"]))
    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "personal_柔軟性": flex.astype(int).values,
        "personal_主体性相談": proactive.astype(int).values,
        "personal_計画性": plan.astype(int).values,
    })


def create_location_match_features(persona_df):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    desired = ws_section.apply(extract_desired_location)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()
    # 希望地を抽出できなかった行は「不明」として区別する（一致でも不一致でもない）
    match_cat = pd.Series("unknown", index=persona_df.index)
    match_cat[desired.notna()] = match[desired.notna()].map({True: "match", False: "mismatch"})
    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "勤務地希望マッチ": match_cat.values,
    })


def first_raise_month(g):
    g = g.sort_values("経過月数")
    salaries = g["月例給与_円"].values
    months = g["経過月数"].values
    base = salaries[0]
    for i in range(1, len(salaries)):
        if salaries[i] > base:
            return months[i]
    return np.nan


def create_raise_timing_features(monthly_df, employee_ids):
    raise_month = monthly_df.groupby("社員ID", group_keys=False).apply(first_raise_month, include_groups=False)
    raise_month = raise_month.reindex(employee_ids)
    early_flag = (raise_month <= 6).astype(int)
    return pd.DataFrame({
        "社員ID": employee_ids,
        "早期昇給フラグ": early_flag.values,
        "初回昇給月": raise_month.values,
    })


def create_major_job_mismatch_features(persona_df):
    is_analytical_major = persona_df["専攻分野"].isin(ANALYTICAL_MAJOR)
    is_analytical_job = persona_df["初期職種"].isin(ANALYTICAL_JOB)

    # 4値カテゴリ（決定木が交互作用を直接学習しやすいよう明示的にエンコード）
    state = pd.Series("", index=persona_df.index)
    state[is_analytical_major & is_analytical_job] = "分析系専攻_分析系職種"
    state[is_analytical_major & ~is_analytical_job] = "分析系専攻_非分析系職種"
    state[~is_analytical_major & is_analytical_job] = "非分析系専攻_分析系職種"
    state[~is_analytical_major & ~is_analytical_job] = "非分析系専攻_非分析系職種"

    # 片方向ミスマッチフラグ（EDA体系探索で確認した最も強いシグナル:
    # 非分析系専攻の社員が分析系職種に配属された場合。逆方向はほぼ無風）
    mismatch_flag = (~is_analytical_major & is_analytical_job).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "専攻職種_適合状態": state.values,
        "専攻職種_分析ミスマッチ": mismatch_flag.values,
    })


def create_momentum_features(monthly_df, employee_ids, metrics, recent_n=3, prior_n=3):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        max_month = emp_data["経過月数"].max()
        features = {"社員ID": employee_id}
        for m in metrics:
            recent = emp_data[emp_data["経過月数"] > max_month - recent_n][m].mean()
            prior = emp_data[
                (emp_data["経過月数"] <= max_month - recent_n)
                & (emp_data["経過月数"] > max_month - recent_n - prior_n)
            ][m].mean()
            features[f"{m}_momentum_diff"] = recent - prior if pd.notna(recent) and pd.notna(prior) else np.nan
            features[f"{m}_momentum_ratio"] = (
                recent / prior if pd.notna(recent) and pd.notna(prior) and prior != 0 else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_density_features(monthly_df, employee_ids):
    agg = monthly_df.groupby("社員ID").agg(
        残業時間_sum=("残業時間", "sum"),
        研修時間_sum=("研修時間", "sum"),
        情報共有件数_sum=("情報共有件数", "sum"),
        面談回数_sum=("上司との面談実施回数", "sum"),
        在宅勤務日数_sum=("在宅勤務日数", "sum"),
        有給取得日数_sum=("有給取得日数", "sum"),
        欠勤日数_sum=("欠勤日数", "sum"),
        担当プロジェクト数_sum=("担当プロジェクト数", "sum"),
        出勤月数=("経過月数", "count"),
        上司ID_nunique=("上司ID", "nunique"),
    ).reindex(employee_ids)

    out = pd.DataFrame(index=agg.index)
    out["残業時間_per_プロジェクト"] = agg["残業時間_sum"] / agg["担当プロジェクト数_sum"].replace(0, np.nan)
    out["情報共有件数_per_出勤月"] = agg["情報共有件数_sum"] / agg["出勤月数"]
    out["面談回数_per_上司数"] = agg["面談回数_sum"] / agg["上司ID_nunique"].replace(0, np.nan)
    out["研修時間_per_残業時間"] = agg["研修時間_sum"] / agg["残業時間_sum"].replace(0, np.nan)
    out["有給取得_per_欠勤"] = agg["有給取得日数_sum"] / (agg["欠勤日数_sum"] + 0.1)
    out["在宅勤務_per_出勤月"] = agg["在宅勤務日数_sum"] / agg["出勤月数"]
    out["担当プロジェクト_per_出勤月"] = agg["担当プロジェクト数_sum"] / agg["出勤月数"]
    out["情報共有_per_面談"] = agg["情報共有件数_sum"] / (agg["面談回数_sum"] + 0.1)
    out = out.replace([np.inf, -np.inf], np.nan).reset_index().rename(columns={"index": "社員ID"})
    return out


print("✅ 却下済みブロックのビルダー関数 定義完了")


✅ 却下済みブロックのビルダー関数 定義完了


## 21. 各ブロックの特徴量を生成（split非依存）

In [44]:
logger.info("=" * 60); logger.info("却下済み/未検証ブロックの特徴量を生成中...")

train_memo       = create_memo_structured_features(train_persona)      # E
test_memo        = create_memo_structured_features(test_persona)
train_selfstudy  = create_self_study_features(train_monthly, train_ids) # G
test_selfstudy   = create_self_study_features(test_monthly, test_ids)
train_engagement = create_engagement_deepdive_features(train_monthly, train_ids)  # H
test_engagement  = create_engagement_deepdive_features(test_monthly, test_ids)
train_personal   = create_personal_impression_features(train_persona)  # I
test_personal    = create_personal_impression_features(test_persona)
train_locmatch   = create_location_match_features(train_persona)       # J
test_locmatch    = create_location_match_features(test_persona)
train_raise      = create_raise_timing_features(train_monthly, train_ids)  # K
test_raise       = create_raise_timing_features(test_monthly, test_ids)
train_majorjob   = create_major_job_mismatch_features(train_persona)   # M
test_majorjob    = create_major_job_mismatch_features(test_persona)
train_momentum   = create_momentum_features(train_monthly, train_ids, MOMENTUM_METRICS)  # N
test_momentum    = create_momentum_features(test_monthly, test_ids, MOMENTUM_METRICS)
train_density    = create_density_features(train_monthly, train_ids)   # O
test_density     = create_density_features(test_monthly, test_ids)

for nm, df_ in [("E_memo", train_memo), ("G_selfstudy", train_selfstudy), ("H_engagement", train_engagement),
                ("I_impression", train_personal), ("J_locmatch", train_locmatch), ("K_raise", train_raise),
                ("M_majorjob", train_majorjob), ("N_momentum", train_momentum), ("O_density", train_density)]:
    logger.info(f"  {nm:14s}: {df_.shape[1]-1:3d}列, {len(df_)}行")
logger.info("ブロック特徴量の生成完了")

[2026-08-12 13:18:17] [INFO] ============================================================


INFO:46_cat_num_split_ensemble:============================================================


[2026-08-12 13:18:17] [INFO] 却下済み/未検証ブロックの特徴量を生成中...


INFO:46_cat_num_split_ensemble:却下済み/未検証ブロックの特徴量を生成中...


[2026-08-12 13:19:52] [INFO]   E_memo        :   4列, 2761行


...

INFO:46_cat_num_split_ensemble:  N_momentum    :  30列, 2761行


[2026-08-12 13:19:52] [INFO]   O_density     :   8列, 2761行


INFO:46_cat_num_split_ensemble:  O_density     :   8列, 2761行


[2026-08-12 13:19:52] [INFO] ブロック特徴量の生成完了


INFO:46_cat_num_split_ensemble:ブロック特徴量の生成完了


## 22. `prepare_split`（39_版: 全ブロック対応）

`37_` 版に、却下済みブロック E/F/G/H/I/J/K/M/N/O のマージ処理を足しただけ。
L_v2・全件学習（`split_ratio=1.0`）・検証セットからの早期退職者除外はそのまま引き継ぐ。

Fブロックだけは他と違い、学習期間のIDのみで線形回帰をfitする必要があるため
`prepare_split` の内部に置いている（`20_`と同一のリーク対策）。

In [45]:
def prepare_split(split_ratio, extra_blocks=None, exclude_early_from_val=True):
    '''指定した分割比率で特徴量を組み立てる。

    28_ からの変更点は2つだけ:
      - split_ratio=1.0 を許容（Train全件学習用。ag_tuningは空になる）
      - exclude_early_from_val=True のとき、検証セットから早期退職者を除く（改善3）
    特徴量の作り方そのものは 28_ と完全に同一。
    '''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L1" in extra_blocks:
        tf = tf.merge(train_reloc_v1, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v1, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")

    if "E" in extra_blocks:
        tf = tf.merge(train_memo, on=ID_COL, how="left")
        ttf = ttf.merge(test_memo, on=ID_COL, how="left")

    if "G" in extra_blocks:
        tf = tf.merge(train_selfstudy, on=ID_COL, how="left")
        ttf = ttf.merge(test_selfstudy, on=ID_COL, how="left")

    if "H" in extra_blocks:
        tf = tf.merge(train_engagement, on=ID_COL, how="left")
        ttf = ttf.merge(test_engagement, on=ID_COL, how="left")

    if "I" in extra_blocks:
        tf = tf.merge(train_personal, on=ID_COL, how="left")
        ttf = ttf.merge(test_personal, on=ID_COL, how="left")

    if "J" in extra_blocks:
        tf = tf.merge(train_locmatch, on=ID_COL, how="left")
        ttf = ttf.merge(test_locmatch, on=ID_COL, how="left")

    if "K" in extra_blocks:
        tf = tf.merge(train_raise, on=ID_COL, how="left")
        ttf = ttf.merge(test_raise, on=ID_COL, how="left")

    if "M" in extra_blocks:
        tf = tf.merge(train_majorjob, on=ID_COL, how="left")
        ttf = ttf.merge(test_majorjob, on=ID_COL, how="left")

    if "N" in extra_blocks:
        tf = tf.merge(train_momentum, on=ID_COL, how="left")
        ttf = ttf.merge(test_momentum, on=ID_COL, how="left")

    if "O" in extra_blocks:
        tf = tf.merge(train_density, on=ID_COL, how="left")
        ttf = ttf.merge(test_density, on=ID_COL, how="left")

    # F: 中途入社者の「前職経験月数から予測される等級」からの残差（20_と同一。学習期間のIDのみでfit）
    if "F" in extra_blocks:
        mid_fit = tf[(tf[ID_COL].isin(train_period_ids)) & (tf["入社区分"] == "中途")]
        reg = LinearRegression().fit(mid_fit[["前職経験月数"]].values, mid_fit["初期等級_num"].values)
        for df_ in [tf, ttf]:
            is_mid = (df_["入社区分"] == "中途")
            df_["経験等級_残差"] = np.nan
            if is_mid.sum() > 0:
                df_.loc[is_mid, "経験等級_残差"] = (
                    df_.loc[is_mid, "初期等級_num"].values - reg.predict(df_.loc[is_mid, ["前職経験月数"]].values)
                )
            df_["is_中途"] = is_mid.astype(int)

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    # --- 改善3: 検証セットから早期退職者を除く（学習側からは除かない） ---
    if exclude_early_from_val and len(ag_tuning) > 0:
        n_before = len(ag_tuning)
        ag_tuning = ag_tuning[~ag_tuning.index.isin(EARLY_LEAVER_IDS)]
        logger.info(f"  検証セット: {n_before} → {len(ag_tuning)}件（早期退職者{n_before - len(ag_tuning)}名を除外）")

    return ag_train, ag_tuning, ttf

print("✅ prepare_split定義完了（39_版: 却下済みブロックE/F/G/H/I/J/K/M/N/Oに対応）")

✅ prepare_split定義完了（39_版: 却下済みブロックE/F/G/H/I/J/K/M/N/Oに対応）


## 24. 基本split（`40_` R0_ref と同一の441列）

In [46]:
# ============================================================
# 基本split（40_ / 37_ と同一）
#   39_ のセル群は prepare_split を定義するだけで、splitの組み立てはしていない。
#   次セルのグループ棚卸しは「基本441列だけのフレーム」を前提にしている
#   （却下ブロックの列が混ざると『どのグループにも属さない列』のassertが落ちる）ので、
#   ここで L2 のみの441列フレームを作っておく。
# ============================================================

BLOCK = {"L2"}   # 28_ の L_v2_extended（40_ R0_ref と同一の441列）

ag_train_80, ag_val_all, test_features = prepare_split(
    0.8, extra_blocks=BLOCK, exclude_early_from_val=False)
ag_train_80b, ag_val_surv, _ = prepare_split(
    0.8, extra_blocks=BLOCK, exclude_early_from_val=True)
ag_full, _ag_empty, test_features_full = prepare_split(
    1.0, extra_blocks=BLOCK, exclude_early_from_val=True)

assert len(_ag_empty) == 0, "全件学習時の検証セットは空のはず"
assert len(ag_train_80) == len(ag_train_80b), "AとB/Cの学習データは同一のはず"
assert len(_feature_cols(ag_train_80)) == 441, \
    f"{len(_feature_cols(ag_train_80))}列（40_ R0_ref と同じ441列のはず）"
print(f"基本441列のsplit: 学習80% {len(ag_train_80b)} / 検証(生存者) {len(ag_val_surv)} / 全件 {len(ag_full)}")
print("✅ 441列を確認（40_ R0_ref と同一）")


[2026-08-12 13:19:53] [INFO]   検証セット: 553 → 535件（早期退職者18名を除外）


INFO:46_cat_num_split_ensemble:  検証セット: 553 → 535件（早期退職者18名を除外）


基本441列のsplit: 学習80% 2208 / 検証(生存者) 535 / 全件 2761
✅ 441列を確認（40_ R0_ref と同一）


## 25. 特徴量グループの棚卸し（`40_` から移植）

In [47]:
# ※ このセルは 40_feature_reduction.ipynb から移植したもの（内容は同一）。

# ============================================================
# 特徴量グループの棚卸し
#   prepare_split() が merge している元フレームごとに列を分類する。
#   「どのグループにも属さない列」「2グループに重複する列」が出たら
#   減量の定義がずれているのでassertで止める。
# ============================================================

ALL_FEATS = set(_feature_cols(ag_train_80))

# prepare_split() 内で生成される派生列（元フレームを持たないのでここに明示）
DERIVED_COLS = [
    "残業時間_mean_job_deviation", "研修時間_mean_job_deviation", "360度評価_親和度_mean_job_deviation",
    "研修時間_職種比", "研修時間_区分比",
    "初任給_等級内偏差", "初任給_区分内偏差", "月例給与_等級内偏差",
]
# 部署Target Encoding が生む列（create_department_target_encoding の出力）
DEPT_TE_COLS = ["dept_target_enc", "dept_size"]


def _cols_of(df):
    return [c for c in df.columns if c != ID_COL]


_RAW_GROUPS = {
    "persona":   [c for c in train_persona.columns if c not in (ID_COL, TARGET_COL)],
    "agg":       _cols_of(train_monthly_agg),
    "catchange": _cols_of(train_cat_change),
    "missing":   _cols_of(train_missing),
    "domain":    _cols_of(train_domain),
    "advstats":  _cols_of(train_advanced_stats),
    "cluster":   _cols_of(train_cluster),
    "deptte":    DEPT_TE_COLS,
    "edafeat":   _cols_of(train_eda_feats),
    "mgr":       _cols_of(train_mgr),
    "quarterly": _cols_of(train_quarterly_exp),
    "tfidf":     [c for _df in tfidf_train_list for c in _cols_of(_df)],
    "L2":        _cols_of(train_reloc_v2),
    "derived":   DERIVED_COLS,
}

# 実際に特徴量として残っている列だけに絞る（drop_colsで消えたものを自動的に除外）
FEATURE_GROUPS = {g: [c for c in cols if c in ALL_FEATS] for g, cols in _RAW_GROUPS.items()}
ALL_GROUPS = set(FEATURE_GROUPS)

_covered = [c for cols in FEATURE_GROUPS.values() for c in cols]
_dupes = sorted({c for c in _covered if _covered.count(c) > 1})
assert not _dupes, f"複数グループに重複している列: {_dupes}"
_orphans = sorted(ALL_FEATS - set(_covered))
assert not _orphans, f"どのグループにも属さない列: {_orphans}"

print(f"特徴量 合計 {len(ALL_FEATS)} 列")
print("-" * 52)
for g in sorted(FEATURE_GROUPS, key=lambda x: -len(FEATURE_GROUPS[x])):
    print(f"  {g:<10s} {len(FEATURE_GROUPS[g]):>4d} 列   例: {FEATURE_GROUPS[g][:2]}")
print("-" * 52)
print("✅ グループ分類は全列を過不足なく覆っている")


特徴量 合計 441 列
----------------------------------------------------
  agg         224 列   例: ['残業時間_mean', '残業時間_std']
  quarterly    80 列   例: ['残業時間_q1_mean_exp', '残業時間_q2_mean_exp']
  tfidf        45 列   例: ['入社時メモ_tfidf_svd_0', '入社時メモ_tfidf_svd_1']
  advstats     25 列   例: ['残業時間_skew', '残業時間_kurtosis']
  persona      21 列   例: ['入社区分', '入社時年齢']
  catchange    14 列   例: ['部署ID_changes', '部署ID_unique_count']
  edafeat      11 列   例: ['欠勤発生月数', '欠勤_最長連続月数']
  derived       8 列   例: ['残業時間_mean_job_deviation', '研修時間_mean_job_deviation']
  missing       4 列   例: ['360度評価_親和度_missing_rate', '360度評価_信頼度_missing_rate']
  domain        3 列   例: ['engagement_score', 'overtime_stability']
  deptte        2 列   例: ['dept_target_enc', 'dept_size']
  L2            2 列   例: ['転居x勤務地_状態_v2', '転居x勤務地_ダブル悪条件_v2']
  cluster       1 列   例: ['cluster']
  mgr           1 列   例: ['初期上司_部下数']
----------------------------------------------------
✅ グループ分類は全列を過不足なく覆っている


In [48]:
# ※ このセルは 40_feature_reduction.ipynb から移植したもの（内容は同一）。

# ============================================================
# 月次集約(agg)の「指標 × 統計」分解
#   create_monthly_aggregation_features が作る 16指標 × 14統計 を分解し、
#   冗長な統計を落とせるようにする。
# ============================================================

AGG_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]
AGG_ALL_STATS = [
    "mean", "std", "min", "max", "median", "cv",
    "early_mean", "mid_mean", "late_mean", "late_minus_early", "late_early_ratio",
    "slope", "diff", "ratio",
]

# 残す統計。冗長性を根拠に選ぶ（検証スコアで選んでいない）:
#   median←mean と重複 / cv←std/mean の比 / min,max←外れ値1点 /
#   mid_mean←early,lateから内挿可能 / late_minus_early,late_early_ratio,diff,ratio←slopeと同義
AGG_KEEP_STATS = {"mean", "std", "early_mean", "late_mean", "slope"}


def _agg_stat(col):
    """agg列名を (指標, 統計) に分解して統計名を返す。最長一致で指標を特定する。"""
    best = None
    for m in AGG_METRICS:
        if col.startswith(m + "_") and (best is None or len(m) > len(best)):
            best = m
    if best is None:
        return None
    return col[len(best) + 1:]


_unmapped = [c for c in FEATURE_GROUPS["agg"] if _agg_stat(c) not in AGG_ALL_STATS]
assert not _unmapped, f"指標×統計に分解できないagg列: {_unmapped}"

AGG_SLIM_COLS = [c for c in FEATURE_GROUPS["agg"] if _agg_stat(c) in AGG_KEEP_STATS]
print(f"agg: {len(FEATURE_GROUPS['agg'])} 列 → 統計を{sorted(AGG_KEEP_STATS)}に限定すると {len(AGG_SLIM_COLS)} 列")
print(f"落とす統計: {sorted(set(AGG_ALL_STATS) - AGG_KEEP_STATS)}")


agg: 224 列 → 統計を['early_mean', 'late_mean', 'mean', 'slope', 'std']に限定すると 80 列
落とす統計: ['cv', 'diff', 'late_early_ratio', 'late_minus_early', 'max', 'median', 'mid_mean', 'min', 'ratio']


## A. 特徴量プールの構築（却下ブロック込み）

In [49]:
# ============================================================
# 第A節: 特徴量プールの構築（却下ブロックを全部入れる）
# ============================================================

ALL_BLOCKS = {"L2", "E", "F", "G", "H", "I", "J", "K", "M", "N", "O"}

logger.info("=" * 60)
logger.info(f"全ブロック込みのsplitを構築: {sorted(ALL_BLOCKS)}")
pool_train_80, pool_val, pool_test = prepare_split(0.8, extra_blocks=ALL_BLOCKS,
                                                   exclude_early_from_val=True)
pool_full, _pool_empty, pool_test_full = prepare_split(1.0, extra_blocks=ALL_BLOCKS,
                                                       exclude_early_from_val=True)
assert len(_pool_empty) == 0, "全件学習時の検証セットは空のはず"
assert list(pool_val.index) == list(ag_val_surv.index), "検証セットが 40_ と一致しない"

POOL = _feature_cols(pool_train_80)
assert POOL == _feature_cols(pool_full), "80%学習と全件でプール列が食い違う"
POOL_CAT = [c for c in POOL if pool_full[c].dtype == "object"]
POOL_NUM = [c for c in POOL if c not in POOL_CAT]

print(f"プール合計 {len(POOL)} 列（カテゴリ {len(POOL_CAT)} / 数値 {len(POOL_NUM)}）")
print(f"  ※ 基本441列 + 却下ブロック = {len(POOL)} 列")
print()
print("カテゴリ列の一覧:")
for c in POOL_CAT:
    print(f"    {c}")


[2026-08-12 13:19:54] [INFO] ============================================================


INFO:46_cat_num_split_ensemble:============================================================


[2026-08-12 13:19:54] [INFO] 全ブロック込みのsplitを構築: ['E', 'F', 'G', 'H', 'I', 'J', 'K', 'L2', 'M', 'N', 'O']


INFO:46_cat_num_split_ensemble:全ブロック込みのsplitを構築: ['E', 'F', 'G', 'H', 'I', 'J', 'K', 'L2', 'M', 'N', 'O']


[2026-08-12 13:19:54] [INFO]   検証セット: 553 → 535件（早期退職者18名を除外）


INFO:46_cat_num_split_ensemble:  検証セット: 553 → 535件（早期退職者18名を除外）


プール合計 499 列（カテゴリ 14 / 数値 485）
  ※ 基本441列 + 却下ブロック = 499 列

カテゴリ列の一覧:
    入社区分
    専攻分野
    採用経路
    性別
    初期職種
    初期勤務地
    初期役割
    転居x勤務地_状態_v2
    memo_career_cat
    memo_転居許容
    memo_在宅希望
    memo_希望勤務地
    勤務地希望マッチ
    専攻職種_適合状態


## B. 参照モデルの113列

In [50]:
# ============================================================
# 第B節: 参照モデルの113列（40_ R6_lean と同一）
# ============================================================

CORE_GROUPS = {"persona", "agg", "deptte", "derived", "L2"}
LEAN_SPEC = {"groups": CORE_GROUPS, "agg_stats": AGG_KEEP_STATS}


def cols_for(spec, df):
    keep = set()
    for g in spec["groups"]:
        if g == "agg" and spec["agg_stats"] is not None:
            keep |= {c for c in FEATURE_GROUPS["agg"] if _agg_stat(c) in spec["agg_stats"]}
        else:
            keep |= set(FEATURE_GROUPS[g])
    return [c for c in _feature_cols(df) if c in keep]


LEAN113 = cols_for(LEAN_SPEC, pool_full)
assert len(LEAN113) == 113, f"{len(LEAN113)}列（40_ R6_lean と同じ113列のはず）"
assert set(LEAN113) <= set(POOL), "113列がプールに含まれていない"
print(f"✅ 参照モデルの特徴量 {len(LEAN113)} 列（40_ R6_lean と同一）")


✅ 参照モデルの特徴量 113 列（40_ R6_lean と同一）


## C. 設定の事前登録

In [51]:
# ============================================================
# 第C節: 設定の事前登録
# ============================================================

A_PARAMS = {
    "depth": 4, "learning_rate": 0.03518359458951149, "l2_leaf_reg": 2.217690447016724,
    "border_count": 218, "bagging_temperature": 0.6787467566574921,
    "random_strength": 1.438494697238285,
}
ITER = 560
SEEDS_SUB = [42, 2024, 7, 1234, 99]
SEEDS_VAL = [42, 2024, 7, 1234, 99, 555, 31337, 2718]

# 重要度による選抜（事前登録）
IMP_CUM_THRESHOLD = 0.95   # 累積重要度がここに達するまで採用
IMP_MIN_COLS = 10
IMP_MAX_COLS = 100

# ゲート（43_ と同一）
GATE_MARGIN = 0.01         # 単体valが M0 から +0.01 以内なら参加
NOISE_FLOOR_P95 = 0.02122  # 41_ 実測のノイズ床。提出判定に使う

print(f"重要度選抜: 累積{IMP_CUM_THRESHOLD:.0%}まで、{IMP_MIN_COLS}〜{IMP_MAX_COLS}列にクリップ")
print(f"参加ゲート : 単体valが M0 + {GATE_MARGIN} 以内")
print(f"提出ゲート : M0との予測平均絶対差 > {NOISE_FLOOR_P95}")


重要度選抜: 累積95%まで、10〜100列にクリップ
参加ゲート : 単体valが M0 + 0.01 以内
提出ゲート : M0との予測平均絶対差 > 0.02122


## D. 重要度による選抜

In [52]:
# ============================================================
# 第D節: 重要度による選抜
#   検証セットを一切使わない（学習データの構造だけから計算する）。
#   40_ のLOO診断（検証スコアで選び 42_ で符号反転）とは性質が違う。
# ============================================================

def _prep(df, feats):
    return df[feats].fillna(-999)


def select_by_importance(feats, tag):
    cats = [c for c in feats if c in POOL_CAT]
    logger.info(f"[{tag}] プール {len(feats)}列 から重要度で選抜（seed=42, 1本）")
    m = cb.CatBoostClassifier(**A_PARAMS, iterations=ITER, random_seed=42, verbose=False,
                              cat_features=cats, task_type="CPU")
    m.fit(_prep(pool_train_80, feats), pool_train_80[TARGET_COL])
    imp = pd.Series(m.get_feature_importance(), index=feats).sort_values(ascending=False)
    total = imp.sum()
    if total <= 0:
        keep = list(imp.index[:IMP_MIN_COLS])
    else:
        cum = (imp / total).cumsum()
        n = int((cum < IMP_CUM_THRESHOLD).sum()) + 1
        n = int(np.clip(n, IMP_MIN_COLS, IMP_MAX_COLS))
        keep = list(imp.index[:n])
    logger.info(f"[{tag}] {len(feats)} → {len(keep)}列（累積重要度 "
                f"{imp[keep].sum() / total if total > 0 else float('nan'):.3f}）")
    imp.to_csv(CHECKPOINT_DIR / f"{SCRIPT_NAME}_importance_{tag}.csv", header=["importance"])
    return [c for c in feats if c in set(keep)], imp   # 元の列順を保つ


SEL_CAT, IMP_CAT = select_by_importance(POOL_CAT, "cat")
SEL_NUM, IMP_NUM = select_by_importance(POOL_NUM, "num")

print(f"カテゴリ: {len(POOL_CAT)} → {len(SEL_CAT)} 列")
print(IMP_CAT.head(15).round(4).to_string())
print()
print(f"数値: {len(POOL_NUM)} → {len(SEL_NUM)} 列")
print(IMP_NUM.head(15).round(4).to_string())

assert len(SEL_CAT) >= IMP_MIN_COLS or len(POOL_CAT) < IMP_MIN_COLS
assert set(SEL_CAT) & set(SEL_NUM) == set(), "カテゴリと数値の選抜が重複している"

MODEL_FEATS = {"M0_lean113": LEAN113, "M1_cat": SEL_CAT, "M2_num": SEL_NUM}
for k, v in MODEL_FEATS.items():
    print(f"  {k:<12s} {len(v):>4d}列")


[2026-08-12 13:19:55] [INFO] [cat] プール 14列 から重要度で選抜（seed=42, 1本）


INFO:46_cat_num_split_ensemble:[cat] プール 14列 から重要度で選抜（seed=42, 1本）


[2026-08-12 13:19:57] [INFO] [cat] 14 → 11列（累積重要度 0.960）


INFO:46_cat_num_split_ensemble:[cat] 14 → 11列（累積重要度 0.960）


[2026-08-12 13:19:57] [INFO] [num] プール 485列 から重要度で選抜（seed=42, 1本）


INFO:46_cat_num_split_ensemble:[num] プール 485列 から重要度で選抜（seed=42, 1本）


[2026-08-12 13:20:01] [INFO] [num] 485 → 100列（累積重要度 0.678）


INFO:46_cat_num_split_ensemble:[num] 485 → 100列（累積重要度 0.678）


カテゴリ: 14 → 11 列
初期職種               18.1091
転居x勤務地_状態_v2       15.2208
専攻分野               13.4123
専攻職種_適合状態          11.4369
初期役割                8.2258
入社区分                6.5165
採用経路                5.5046
memo_career_cat     5.3188
初期勤務地               5.1555
memo_在宅希望           3.8506
memo_転居許容           3.2249
memo_希望勤務地          2.7432
性別                  0.8585
勤務地希望マッチ            0.4226

数値: 485 → 100 列
転居x勤務地_ダブル悪条件_v2            6.5900
専攻職種_分析ミスマッチ                3.9422
残業時間_min                    2.7620
自己学習ユニークテーマ数                2.1911
残業時間_cv                     1.7272
残業時間_median                 1.5116
上司からのフィードバック_tfidf_svd_1    1.5027
残業時間_mid_mean               1.4821
残業時間_q3_mean_exp            1.4666
担当プロジェクト数_cv                1.3573
入社時メモ_tfidf_svd_1           1.2459
残業時間_q2_mean_exp            1.2318
情報共有_per_面談                 1.0840
残業時間_late_mean              1.0290
上司との面談実施回数_std              0.9124
  M0_lean113    113列
  M1_cat         11列
  M2_num        100列


## E. 3モデルの学習と多様性チェック

In [53]:
# ============================================================
# 第E節: 3モデルの学習
# ============================================================

PRED_DIR = CHECKPOINT_DIR / SCRIPT_NAME
PRED_DIR.mkdir(parents=True, exist_ok=True)

y_val = pool_val[TARGET_COL].values
SINGLE = {}
for name, feats in MODEL_FEATS.items():
    vpath, tpath = PRED_DIR / f"{name}_val.npy", PRED_DIR / f"{name}_test.npy"
    if vpath.exists() and tpath.exists():
        vps, tps = np.load(vpath), np.load(tpath)
        assert vps.shape == (len(SEEDS_VAL), len(y_val)), f"{name}: valの形が不正 {vps.shape}"
        assert tps.shape == (len(SEEDS_SUB), len(pool_test_full)), f"{name}: testの形が不正 {tps.shape}"
        logger.info(f"[{name}] 保存済み予測から復元")
    else:
        cats = [c for c in feats if c in POOL_CAT]
        logger.info("=" * 60)
        logger.info(f"[{name}] {len(feats)}列（カテゴリ {len(cats)}）")
        Xtr, ytr = _prep(pool_train_80, feats), pool_train_80[TARGET_COL]
        Xva = _prep(pool_val, feats)
        vps = np.array([cb.CatBoostClassifier(**A_PARAMS, iterations=ITER, random_seed=s,
                                              verbose=False, cat_features=cats, task_type="CPU")
                        .fit(Xtr, ytr).predict_proba(Xva)[:, 1] for s in SEEDS_VAL])
        Xfu, yfu = _prep(pool_full, feats), pool_full[TARGET_COL]
        Xte = _prep(pool_test_full, feats)
        tps = np.array([cb.CatBoostClassifier(**A_PARAMS, iterations=ITER, random_seed=s,
                                              verbose=False, cat_features=cats, task_type="CPU")
                        .fit(Xfu, yfu).predict_proba(Xte)[:, 1] for s in SEEDS_SUB])
        np.save(vpath, vps); np.save(tpath, tps)
    singles = [log_loss(y_val, v) for v in vps]
    SINGLE[name] = {"n_features": len(feats),
                    "val_preds": vps.mean(axis=0), "test_preds": tps.mean(axis=0),
                    "test_by_seed": tps,
                    "val_seedavg": float(log_loss(y_val, vps.mean(axis=0))),
                    "val_single_mean": float(np.mean(singles)),
                    "val_single_sd": float(np.std(singles))}
    np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{name}_valpreds.npy", vps)
    logger.info(f"  val シード平均 {SINGLE[name]['val_seedavg']:.6f}")

m0 = SINGLE["M0_lean113"]["val_seedavg"]
print(f"{'model':<12s}{'列数':>5s}{'val(8シード)':>14s}{'単一sd':>10s}{'M0差':>11s}{'ゲート':>8s}")
print("-" * 62)
GATE_PASS = []
for n, r in SINGLE.items():
    d = r["val_seedavg"] - m0
    ok = (n == "M0_lean113") or (d <= GATE_MARGIN)
    if ok:
        GATE_PASS.append(n)
    print(f"{n:<12s}{r['n_features']:>5d}{r['val_seedavg']:>14.6f}{r['val_single_sd']:>10.6f}"
          f"{d:>+11.6f}{'✅' if ok else '❌':>8s}")
print(f"\nゲート通過: {GATE_PASS}")
for n, r in SINGLE.items():
    if n == "M0_lean113":
        continue
    d = r["val_seedavg"] - m0
    if abs(d - GATE_MARGIN) < 0.001:
        print(f"  ⚠️ {n}: M0差 {d:+.6f} はゲート {GATE_MARGIN} の境界から {abs(d-GATE_MARGIN):.6f} しか離れていない。")
        print(f"     この判定は実質コイン投げ。多様性（相関）と併せて判断すること。")

print()
print("=" * 62)
print("多様性: Test予測の相関")
print("=" * 62)
names = list(SINGLE)
print(pd.DataFrame({a: {b: np.corrcoef(SINGLE[a]["test_preds"], SINGLE[b]["test_preds"])[0, 1]
                        for b in names} for a in names}).round(5).to_string())
_m0 = SINGLE["M0_lean113"]["test_by_seed"]
print(f"\n  参考: M0内のシード間相関（多様性ゼロの基準） "
      f"{np.mean([np.corrcoef(_m0[i], _m0[j])[0, 1] for i in range(5) for j in range(i + 1, 5)]):.5f}")


[2026-08-12 13:20:01] [INFO] ============================================================


INFO:46_cat_num_split_ensemble:============================================================


[2026-08-12 13:20:01] [INFO] [M0_lean113] 113列（カテゴリ 8）


INFO:46_cat_num_split_ensemble:[M0_lean113] 113列（カテゴリ 8）


[2026-08-12 13:20:36] [INFO]   val シード平均 0.514642


...

[2026-08-12 13:21:03] [INFO] [M2_num] 100列（カテゴリ 0）


INFO:46_cat_num_split_ensemble:[M2_num] 100列（カテゴリ 0）


[2026-08-12 13:21:22] [INFO]   val シード平均 0.534594


INFO:46_cat_num_split_ensemble:  val シード平均 0.534594


model          列数     val(8シード)      単一sd        M0差     ゲート
--------------------------------------------------------------
M0_lean113    113      0.514642  0.005220  +0.000000       ✅
M1_cat         11      0.541764  0.002048  +0.027121       ❌
M2_num        100      0.534594  0.002887  +0.019952       ❌

ゲート通過: ['M0_lean113']

多様性: Test予測の相関
            M0_lean113   M1_cat   M2_num
M0_lean113     1.00000  0.79205  0.86335
M1_cat         0.79205  1.00000  0.63082
M2_num         0.86335  0.63082  1.00000

  参考: M0内のシード間相関（多様性ゼロの基準） 0.98575


## F. 等重み平均

In [54]:
# ============================================================
# 第F節: 等重み平均（重みは学習しない）
# ============================================================

RESULT_SCHEMA = ["config", "members", "n_models", "n_features", "val_seedavg",
                 "pred_mean", "mad_vs_M0", "corr_vs_M0", "submission_path"]

COMBOS = {"E_M0_M1": ["M0_lean113", "M1_cat"],
          "E_M0_M2": ["M0_lean113", "M2_num"],
          "E_M1_M2": ["M1_cat", "M2_num"],
          "E_ALL3":  ["M0_lean113", "M1_cat", "M2_num"]}

base_t = SINGLE["M0_lean113"]["test_preds"]
results = {}


def _row(label, members, path):
    v = np.mean([SINGLE[m]["val_preds"] for m in members], axis=0)
    t = np.mean([SINGLE[m]["test_preds"] for m in members], axis=0)
    return make_row(config=label, members=",".join(members), n_models=len(members),
                    n_features=int(sum(SINGLE[m]["n_features"] for m in members)),
                    val_seedavg=float(log_loss(y_val, v)), pred_mean=float(t.mean()),
                    mad_vs_M0=float(np.abs(t - base_t).mean()),
                    corr_vs_M0=float(np.corrcoef(t, base_t)[0, 1]), submission_path=path)


# 参照（M0）は 40_ R6_lean と同一内容。再現チェック用に書き出す
results["M0_lean113"] = _row("M0_lean113", ["M0_lean113"],
                             save_submission(pool_test_full.index, base_t, "M0_lean113"))

for label, members in COMBOS.items():
    if not all(m in GATE_PASS for m in members):
        logger.info(f"[{label}] ゲート未通過のメンバーを含むため作らない: {members}")
        continue
    t = np.mean([SINGLE[m]["test_preds"] for m in members], axis=0)
    results[label] = _row(label, members, save_submission(pool_test_full.index, t, label))

summary = pd.DataFrame(list(results.values()))
print(summary[["config", "members", "n_features", "val_seedavg", "pred_mean",
               "mad_vs_M0", "corr_vs_M0"]].round(6).to_string(index=False))
summary.to_csv(CHECKPOINT_PATH, index=False)

print()
print("=" * 74)
print("【参考】ゲートを無視して等重み平均していたらどうなったか")
print("=" * 74)
print(f"  {'構成':<12s}{'val':>10s}{'M0差':>11s}{'Test平均絶対差':>15s}")
print("  " + "-" * 48)
for label, members in COMBOS.items():
    v = np.mean([SINGLE[m]["val_preds"] for m in members], axis=0)
    t = np.mean([SINGLE[m]["test_preds"] for m in members], axis=0)
    print(f"  {label:<12s}{log_loss(y_val, v):>10.6f}{log_loss(y_val, v) - m0:>+11.6f}"
          f"{np.abs(t - base_t).mean():>15.5f}")
print("\n  ゲートで作らなかった構成もここで値が分かるので、失敗しても情報は残る。")


[2026-08-12 13:21:23] [INFO]   提出ファイル: 20260812_46_cat_num_split_ensemble_M0_lean113.csv（予測平均=0.5874）


INFO:46_cat_num_split_ensemble:  提出ファイル: 20260812_46_cat_num_split_ensemble_M0_lean113.csv（予測平均=0.5874）


[2026-08-12 13:21:23] [INFO] [E_M0_M1] ゲート未通過のメンバーを含むため作らない: ['M0_lean113', 'M1_cat']


INFO:46_cat_num_split_ensemble:[E_M0_M1] ゲート未通過のメンバーを含むため作らない: ['M0_lean113', 'M1_cat']


[2026-08-12 13:21:23] [INFO] [E_M0_M2] ゲート未通過のメンバーを含むため作らない: ['M0_lean113', 'M2_num']


...

[2026-08-12 13:21:23] [INFO] [E_M1_M2] ゲート未通過のメンバーを含むため作らない: ['M1_cat', 'M2_num']


INFO:46_cat_num_split_ensemble:[E_M1_M2] ゲート未通過のメンバーを含むため作らない: ['M1_cat', 'M2_num']


[2026-08-12 13:21:23] [INFO] [E_ALL3] ゲート未通過のメンバーを含むため作らない: ['M0_lean113', 'M1_cat', 'M2_num']


INFO:46_cat_num_split_ensemble:[E_ALL3] ゲート未通過のメンバーを含むため作らない: ['M0_lean113', 'M1_cat', 'M2_num']


    config    members  n_features  val_seedavg  pred_mean  mad_vs_M0  corr_vs_M0
M0_lean113 M0_lean113         113     0.514642   0.587364        0.0         1.0

【参考】ゲートを無視して等重み平均していたらどうなったか
  構成                 val        M0差      Test平均絶対差
  ------------------------------------------------
  E_M0_M1       0.512521  -0.002121        0.05544
  E_M0_M2       0.514612  -0.000030        0.04889
  E_M1_M2       0.516701  +0.002059        0.08347
  E_ALL3        0.509683  -0.004959        0.05565

  ゲートで作らなかった構成もここで値が分かるので、失敗しても情報は残る。


In [55]:
# ============================================================
# M0 の再現性チェック
# ============================================================

_m0f = pd.read_csv(results["M0_lean113"]["submission_path"], header=None, names=[ID_COL, "pred"])
_r6 = sorted((PROJECT_ROOT / "data" / "output").glob("*/*_40_feature_reduction_R6_lean.csv"))
if _r6:
    _r = pd.read_csv(_r6[-1], header=None, names=[ID_COL, "pred"])
    _m = _m0f.merge(_r, on=ID_COL, suffixes=("_m0", "_r6"))
    assert len(_m) == len(_m0f), "社員IDが一致しない"
    _c, _mad = _m["pred_m0"].corr(_m["pred_r6"]), (_m["pred_m0"] - _m["pred_r6"]).abs().mean()
    print(f"R6_lean: {_r6[-1].name}")
    print(f"  相関 {_c:.6f} / 平均絶対差 {_mad:.6f}")
    print("✅ 再現できている" if _c > 0.9999 and _mad < 0.001
          else "⚠️ 再現できていない。プールの構築が113列の値を変えていないか確認すること")
else:
    print("⚠️ R6_leanの提出ファイルが見つからなかった")


R6_lean: 20260811_40_feature_reduction_R6_lean.csv
  相関 1.000000 / 平均絶対差 0.000000
✅ 再現できている


## G. 提出判定

In [56]:
# ============================================================
# 第G節: 提出判定
# ============================================================

summary["提出"] = "見送り"
summary.loc[summary["config"] == "M0_lean113", "提出"] = "不要（提出済み・参照用）"
_ens = summary["config"].str.startswith("E_")
summary.loc[_ens & (summary["mad_vs_M0"] > NOISE_FLOOR_P95), "提出"] = "提出する"
summary.loc[_ens & (summary["mad_vs_M0"] <= NOISE_FLOOR_P95), "提出"] = \
    "見送り（M0との差がノイズ床以下）"

pd.set_option("display.width", 230)
print(summary[["config", "members", "n_features", "val_seedavg", "pred_mean",
               "mad_vs_M0", "corr_vs_M0", "提出"]].round(6).to_string(index=False))
summary.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_summary.csv", index=False)

print()
print("=" * 74)
print("提出候補（メンバー数の少ない順＝現最良からの変更が小さい順）")
print("=" * 74)
_sub = summary[summary["提出"] == "提出する"].sort_values("n_models")
if len(_sub) == 0:
    print("  なし。")
    print("  → 入力を分割して多様性を作っても、CatBoost単体を超える組み合わせは作れなかった。")
    print("     43_（モデルファミリでの多様性）と合わせて、アンサンブルの線は打ち切ってよい。")
else:
    for i, r in enumerate(_sub.itertuples(), 1):
        print(f"{i}. {r.config:<12s} {Path(r.submission_path).name}")
        print(f"     メンバー {r.members} / 計{int(r.n_features)}列 / "
              f"M0との相関 {getattr(r, 'corr_vs_M0'):.5f} / 平均絶対差 {getattr(r, 'mad_vs_M0'):.5f}")
print()
print(f"※ ノイズ床（41_実測）: 95%上限 {NOISE_FLOOR_P95}")


    config    members  n_features  val_seedavg  pred_mean  mad_vs_M0  corr_vs_M0           提出
M0_lean113 M0_lean113         113     0.514642   0.587364        0.0         1.0 不要（提出済み・参照用）

提出候補（メンバー数の少ない順＝現最良からの変更が小さい順）
  なし。
  → 入力を分割して多様性を作っても、CatBoost単体を超える組み合わせは作れなかった。
     43_（モデルファミリでの多様性）と合わせて、アンサンブルの線は打ち切ってよい。

※ ノイズ床（41_実測）: 95%上限 0.02122


## H. 提出方針と結果の解釈

### 提出するファイル

第G節で「提出する」となったものを、**メンバー数の少ない順**（現最良からの変更が小さい順）。
`M0_lean113` は `40_` R6_lean と同一内容なので**提出しない**。
`M1_cat`・`M2_num` の単体も**事前登録どおり提出しない**（材料の質を測るためだけに学習した）。

### 結果の解釈ルール（事前登録）

- **Public < 0.521729** → 入力分割による多様性は有効。次はメンバーの粒度を検討する
  （**重みの学習はしない**。[ensemble-oof-overfitting]）
- **Public ≒ 0.5217 ± 0.001** → 差なし。CatBoost単体（113列）を基準構成のまま維持
- **Public > 0.5217** → 有害。`43_` と合わせてアンサンブルの線を打ち切る

### `M1_cat` / `M2_num` の単体スコアから分かること

- **`M1_cat` が予想外に強い**（M0から+0.02以内）なら、`43_` の解釈
  （この問題ではカテゴリ変数が支配的で、CatBoostのordered target statisticsが効いている）が補強される
- **`M2_num` の方が強い**なら、逆に月次集約の数値情報が主役ということになり、
  `42_` の `S1_no_agg33` が大きく悪化した（+0.0098）ことと整合する
- どちらの結果でも、**却下ブロックの中に「小さいモデルの中でなら効く」列があるか**が
  重要度ランキングから読み取れる

### やらないこと

- **重みを学習しない。** 等重み固定のみ
- **重要度の閾値を結果を見てから変えない**（累積95%・10〜100列は事前登録）
- **検証スコアで提出構成を選び直さない**（分解能±0.011）

### 重要度による選抜について

この重要度（`PredictionValuesChange`）は**検証セットを一切使わず、学習データの構造だけ**から
計算される。`40_` のLOO診断（検証スコアで選び、`42_` の Public で符号反転した）とは性質が違う。
ただし「学習データに適合した基準で選んでいる」ことに変わりはないので、
**採否は Public でのみ判断する**という原則は同じである。

### 関連

- 現最良: `data/output/20260811/20260811_40_feature_reduction_R6_lean.csv`（Public 0.521729）
- モデルファミリでの多様性: `submit_result_report.md` 第67節（`43_`）
- 却下ブロックの一覧と過去の判定: 同 第55〜60節（`39_`）
